<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/IndicF5_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IndicF5 — High-Quality TTS for 11 Indian Languages

400M-parameter polyglot TTS by AI4Bharat (IIT Madras), trained on 1417 hours of speech from Rasa, IndicTTS, LIMMITS, and IndicVoices-R. Supports **11 Indian languages** (Assamese, Bengali, Gujarati, Hindi, Kannada, Malayalam, Marathi, Odia, Punjabi, Tamil, Telugu) and zero-shot voice cloning from a 5–10s reference clip in any supported language.

### Quick Start
1. Connect to a GPU runtime (T4 16 GB works, any tier is fine — model is small)
2. **You need a Hugging Face account** with access to the gated `ai4bharat/IndicF5` weights. Accept the terms at https://huggingface.co/ai4bharat/IndicF5, then add an `HF_TOKEN` Colab secret (key icon in the left sidebar) with a read-access token from https://huggingface.co/settings/tokens
3. Run Steps 1–4 in order
4. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Status |
|-----|------|--------|
|-----|------|--------|
| A100 / L4 / T4 | 8 GB+ | All work — model is small (~800 MB in bf16) |

**How to use:** Pick a language, type the text you want spoken in that language's native script, then either upload a short reference clip (with its transcript) for voice cloning or use one of the preloaded example voices.

**License:** MIT. The model card requires you to only clone voices for which you have explicit permission. Unauthorized voice cloning is strictly prohibited.

In [ ]:
# @title STEP 1 — Install Dependencies and Authenticate# @markdown Installs the `IndicF5` package via git, plus Gradio. Reads `HF_TOKEN` from Colab secrets to access the gated model weights.import os, sys, subprocessimport torchif not torch.cuda.is_available():    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime \u2192 Change runtime type \u2192 T4 / L4 / A100')gpu_name = torch.cuda.get_device_name(0)vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB')# HF token from Colab secrets (key icon in the left sidebar)try:    from google.colab import userdata    HF_TOKEN = userdata.get('HF_TOKEN')    if not HF_TOKEN:        raise ValueError('HF_TOKEN not set')    os.environ['HF_TOKEN'] = HF_TOKEN    os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN    print('[HF] HF_TOKEN loaded from Colab secrets.')except Exception as e:    print()    print('\u274c Could not read HF_TOKEN from Colab secrets.')    print('   IndicF5 weights are gated. To use this notebook:')    print('   1. Create a read-access token at https://huggingface.co/settings/tokens')    print('   2. Accept the model terms at https://huggingface.co/ai4bharat/IndicF5')    print('   3. Add the token as a Colab secret named HF_TOKEN (key icon in the left sidebar)')    print('   4. Re-run this cell.')    raise SystemExit(str(e))print('[pip] Installing IndicF5 from GitHub ...')subprocess.run(['pip', 'install', '-q', '-U', 'git+https://github.com/ai4bharat/IndicF5.git'], check=True)try:    import IndicF5  # noqa: F401    print('  IndicF5 package installed.')except ImportError:    # The package may not expose a top-level module; that's OK if from_pretrained works below    print('  IndicF5 package installed (no top-level module — the AutoModel is loaded from the HF repo).')print('[pip] Installing Gradio + audio tools ...')subprocess.run(['pip', 'install', '-q', '-U', 'gradio==5.33.0', 'soundfile', 'librosa', 'numpy', 'tqdm==4.66.5'], check=True)os.makedirs('/content/audio_out', exist_ok=True)os.makedirs('/content/ref_audio', exist_ok=True)print('[Setup] /content/audio_out and /content/ref_audio ready.')

In [ ]:
# @title STEP 2 — Pre-cache Models and Reference Prompts
# @markdown Downloads the 400M-parameter IndicF5 weights (~800 MB) and 7 bundled reference voice clips for one-click demos.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

REPO = 'ai4bharat/IndicF5'
print(f'[Download] {REPO} (~800 MB) ...')
snapshot_download(REPO)
print('[Download] IndicF5 weights cached.')

# Also pre-cache the 7 example prompt audio clips that ship with the GitHub repo
print('[Download] Bundled reference voice clips from the IndicF5 GitHub repo ...')
PROMPTS = [
    'KAN_F_HAPPY_00001.wav',
    'MAR_F_HAPPY_00001.wav',
    'MAR_F_WIKI_00001.wav',
    'MAR_M_WIKI_00001.wav',
    'PAN_F_HAPPY_00001.wav',
    'PAN_F_HAPPY_00002.wav',
    'TAM_F_HAPPY_00001.wav',
]
for name in PROMPTS:
    url = f'https://github.com/AI4Bharat/IndicF5/raw/main/prompts/{name}'
    out = f'/content/ref_audio/{name}'
    if not os.path.exists(out):
        subprocess.run(['curl', '-sL', '-o', out, url], check=True)
    print(f'  \u2192 {name}')
print('[Download] All bundled voice prompts ready.')

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)
# @markdown Wraps IndicF5's `AutoModel`. Loads on first use, then stays in memory.

import os, time, subprocess, tempfile
import numpy as np
import torch
import soundfile as sf

REPO = 'ai4bharat/IndicF5'
SAMPLE_RATE = 24000
_MODEL = None

def get_model():
    global _MODEL
    if _MODEL is not None:
        return _MODEL
    from transformers import AutoModel
    print(f'[Load] {REPO} (bf16) ...')
    t0 = time.time()
    _MODEL = AutoModel.from_pretrained(REPO, trust_remote_code=True).to('cuda')
    print(f'[Load] Ready in {time.time()-t0:.1f}s')
    return _MODEL

def _materialize_ref_audio(ref_audio_path: str) -> str:
    """Gradio gives us (sample_rate, np.ndarray) tuples when type='numpy'. Save to a temp .wav the model can read."""
    if isinstance(ref_audio_path, tuple) and len(ref_audio_path) == 2:
        sr, audio = ref_audio_path
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        sf.write(tmp.name, audio, sr, format='WAV')
        tmp.flush()
        return tmp.name
    return ref_audio_path  # already a file path

def synth(text: str,
          ref_audio_path,
          ref_text: str,
          out_name: str = 'indicf5.wav') -> tuple[str, int]:
    if not text or not text.strip():
        raise RuntimeError('Text is empty.')
    if not ref_audio_path:
        raise RuntimeError('Upload a reference audio clip — IndicF5 only supports voice cloning.')
    if not ref_text or not ref_text.strip():
        raise RuntimeError('Provide the transcript of the reference audio — it materially improves quality.')

    m = get_model()
    path = _materialize_ref_audio(ref_audio_path)
    audio = m(text.strip(), ref_audio_path=path, ref_text=ref_text.strip())
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    audio = np.asarray(audio, dtype=np.float32)

    out_path = os.path.join('/content/audio_out', out_name)
    sf.write(out_path, audio, SAMPLE_RATE)
    # If we created a temp file, clean it up
    if path != ref_audio_path:
        try:
            os.unlink(path)
        except OSError:
            pass
    return out_path, SAMPLE_RATE

print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app with preloaded example voices in 4 languages. Pick a voice, type text in the target language, hit Generate. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr
import soundfile as sf

def _err(msg):
    return None, f'\u274c {msg}'

def ui_synth(text, ref_audio, ref_text, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some text first.')
    if ref_audio is None:
        return _err('Upload a reference voice clip, or pick one from the Examples below.')
    if not ref_text or not ref_text.strip():
        return _err('Provide the transcript of the reference audio — it materially improves quality.')
    try:
        progress(0.1, desc='Loading model...')
        get_model()
        progress(0.3, desc='Synthesizing...')
        path, sr = synth(text.strip(), ref_audio, ref_text, out_name='ui.wav')
        progress(1.0, desc='Done.')
        return path, f'\u2705 Generated @ {sr} Hz'
    except Exception as e:
        return None, f'\u274c {e}'

# Preloaded examples lifted from the official IndicF5 HF Space (https://huggingface.co/spaces/ai4bharat/IndicF5)
EXAMPLES = [
    {
        'name': 'Punjabi Female (Happy) \u2192 Hindi synthesis',
        'prompt': '/content/ref_audio/PAN_F_HAPPY_00002.wav',
        'ref_text': '\u0a07\u0a71\u0a15 \u0a17\u0a4d\u0a30\u0a3e\u0a39\u0a15 \u0a28\u0a47 \u0a38\u0a3e\u0a21\u0a40 \u0a2c\u0a47\u0a2e\u0a3f\u0a38\u0a3e\u0a32 \u0a38\u0a47\u0a35\u0a3e \u0a2c\u0a3e\u0a30\u0a47 \u0a26\u0a3f\u0a32\u0a4b\u0a02\u0a17\u0a35\u0a3e\u0a39\u0a40 \u0a26\u0a3f\u0a71\u0a24\u0a40 \u0a1c\u0a3f\u0a38 \u0a28\u0a3e\u0a32 \u0a38\u0a3e\u0a28\u0a42\u0a02 \u0a05\u0a28\u0a70\u0a26 \u0a2e\u0a39\u0a3f\u0a38\u0a42\u0a38 \u0a39\u0a4b\u0a07\u0a06\u0961',
        'synth_text': '\u092e\u0948\u0902 \u092c\u093f\u0928\u093e \u0915\u093f\u0938\u0940 \u091a\u093f\u0902\u0924\u093e \u0915\u0947 \u0905\u092a\u0928\u0947 \u0926\u094b\u0938\u094d\u0924\u094b\u0902 \u0915\u094b \u0905\u092a\u0928\u0947 \u0911\u091f\u094b\u092e\u094b\u092c\u093e\u0907\u0932 \u090f\u0915\u094d\u0938\u092a\u0930\u094d\u091f \u0915\u0947 \u092a\u093e\u0938 \u092d\u0947\u091c \u0926\u0947\u0924\u093e \u0939\u0942\u0901 \u0915\u094d\u092f\u094b\u0902\u0915\u093f \u092e\u0948\u0902 \u091c\u093e\u0928\u0924\u093e \u0939\u0942\u0902 \u0915\u093f \u0935\u0939 \u0928\u093f\u0936\u094d\u091a\u093f\u0924 \u0930\u0942\u092a \u0938\u0947 \u0909\u0928\u0915\u0940 \u0938\u092d\u0940 \u091c\u0930\u0942\u0930\u0924\u094b\u0902 \u092a\u0930 \u0916\u0930\u093e \u0909\u0924\u0930\u0947\u0917\u093e\u0961',
    },
    {
        'name': 'Tamil Female (Happy) \u2192 Malayalam synthesis',
        'prompt': '/content/ref_audio/TAM_F_HAPPY_00001.wav',
        'ref_text': '\u0ba8\u0bbe\u0ba9\u0bcd \u0ba8\u0bc6\u0ba9\u0b9a\u0bcd\u0b9a \u0bae\u0bbe\u0ba4\u0bbf\u0bb0\u0bbf\u0baf\u0bc7 \u0b85\u0bae\u0bc7\u0b9a\u0bbe\u0ba9\u0bcd\u0bb2\u0bcd \u0baa\u0bc6\u0bb0\u0bbf\u0baf \u0ba4\u0bb3\u0bcd\u0bb3\u0bc1\u0baa\u0b9f\u0bbf \u0bb5\u0b9f\u0bcd\u0b9f\u0bbf\u0bb0\u0bc1\u0b95\u0bcd\u0b95\u0bc1. \u0b95\u0bae\u0bcd\u0bae\u0bbf \u0b95\u0bbe\u0b9a\u0bc1\u0b95\u0bcd\u0b95\u0bc7\u0baf\u0bc7 \u0b85\u0ba8\u0bcd\u0ba4\u0baa\u0bc1 \u0baa\u0bc1\u0ba4\u0bc1 \u0b9a\u0bc7\u0bae\u0bcd\u0b9a\u0ba9\u0bcd\u0b97\u0bcd \u0bae\u0bbe\u0b9f\u0bb2 \u0bb5\u0bbe\u0b99\u0bcd\u0b95\u0bbf\u0b9f\u0bb2\u0bbe\u0bae\u0bcd.',
        'synth_text': '\u0d2d\u0d15\u0d4d\u0d37\u0d23\u0d24\u0d4d\u0d24\u0d3f\u0d28\u0d4d \u0d36\u0d47\u0d37\u0d02 \u0d24\u0d48\u0d30\u0d4d \u0d38\u0d3e\u0d26\u0d02 \u0d15\u0d34\u0d3f\u0d1a\u0d4d\u0d1a\u0d3e\u0d7d \u0d09\u0d7c\u0d36\u0d3e\u0d30\u0d3e\u0d23\u0d4d\u0d1f\u0d4b \u0d2e\u0d28\u0d4d \u0d16\u0d41\u0d2c\u0d3e\u0d2f\u0d3f \u0d35\u0d30\u0d41\u0d28\u0d4d\u0d24\u0d4d \u0d08\u0d30\u0d41\u0d24\u0d4d\u0d24\u0d4b!',
    },
    {
        'name': 'Marathi Female (Wiki) \u2192 Marathi synthesis',
        'prompt': '/content/ref_audio/MAR_F_WIKI_00001.wav',
        'ref_text': '\u0926\u093f\u0917\u0902\u0924\u0930\u093e\u0935\u094d\u0926\u093e\u0930\u0947 \u0905\u0902\u0924\u0930\u093e\u0933 \u0915\u0915\u094d\u0937\u0947\u0924\u0932\u093e \u0915\u091a\u0930\u093e \u091a\u093f\u0928\u094d\u0939\u093f\u0924 \u0915\u0930\u0923\u094d\u092f\u093e\u0938\u093e\u0920\u0940 \u092a\u094d\u0930\u092f\u0924\u094d\u0928 \u0915\u0947\u0932\u0947 \u091c\u093e\u0924\u094b\u0964',
        'synth_text': '\u092a\u094d\u0930\u093e\u0930\u0902\u092d\u093f\u0915 \u0905\u0902\u0915\u0941\u0930 \u091b\u0947\u0926\u0915. \u092e\u0940 \u0938\u094b\u0932\u093e\u092a\u0942\u0930 \u091c\u093f\u0932\u094d\u0939\u094d\u092f\u093e\u0924\u0940\u0932 \u092e\u093e\u0933\u0936\u093f\u0930\u0938 \u0924\u093e\u0932\u0941\u0915\u094d\u092f\u093e\u0924\u0940\u0932 \u0936\u0947\u0924\u0915\u0930\u0940 \u0917\u0923\u092a\u0924 \u092a\u093e\u091f\u0940\u0932 \u092c\u094b\u0932\u0924\u094b\u092f. \u092e\u093e\u091d\u094d\u092f\u093e \u0909\u0938 \u092a\u093f\u0915\u093e\u0935\u0930 \u092a\u094d\u0930\u093e\u0930\u0902\u092d\u093f\u0915 \u0905\u0902\u0915\u0941\u0930 \u091b\u0947\u0926\u0915 \u0915\u0940\u0921 \u0906\u0922\u0923\u094d\u092f\u093e\u0938 \u092f\u0947\u0924. \u0915\u094d\u0932\u094b\u0930\u0948\u0902\u091f\u094d\u0930\u093e\u0928\u093f\u0932\u0940\u092a\u094d\u0930\u094b\u0932 \u0915\u094b\u0930\u093e\u091c\u0947\u0928) \u0935\u093e\u092a\u0930\u0923\u0947 \u092f\u094b\u0917\u094d\u092f \u0906\u0939\u0947 \u0915\u093e? \u0924\u094d\u092f\u093e\u091a\u0947 \u092a\u094d\u0930\u092e\u093e\u0923 \u0915\u093f\u0924\u0940 \u0905\u0938\u093e\u0935\u0947?',
    },
    {
        'name': 'Kannada Female (Happy) \u2192 Bengali synthesis',
        'prompt': '/content/ref_audio/KAN_F_HAPPY_00001.wav',
        'ref_text': '\u0c92\u0c82\u0ca6\u0cc1 \u0cab\u0ccd\u0cb0\u0cbf\u0c9c\u0ccd\u0c9c\u0cb2\u0ccd\u0cb2\u0cbf  \u0c95\u0cc2\u0cb2\u0cbf\u0c82\u0c97\u0ccd\u200d  \u0cb8\u0cae\u0cb8\u0ccd\u0caf\u0947 \u0c86\u0c97\u0cbf \u0ca8\u0cbe\u0ca8\u0ccd‌  \u0cad\u0cbe\u0cb3 \u0ca6\u0cbf\u0ca8\u0ca6\u0cbf\u0c82\u0ca6 \u0c92\u0ca6\u0ccd\u0ca6\u0cbe\u0ca1\u0ccd\u0c9f\u0cbf\u0ca6\u0ccd\u0ca6\u0cc0\u0ca8\u0cbf, \u0c86\u0ca6\u0cb0\u0947 \u0c85\u0ca6\u0ccd\u0ca8\u0cc0\u0c97 \u0cae\u0947\u0c95\u0cbe\u0ca8\u0cbf\u0c95\u0ccd \u0c86\u0c97\u0cbf\u0cb0\u094b \u0ca8\u0cbf\u0cae\u0ccd\u200d \u0cb8\u0cb9\u0cbe\u0caf\u0c95\u0ccd\u0ca6\u0cbf\u0c82\u0ca6 \u0cac\u0c97\u0947\u0cb9\u0930\u0cbf\u0cb8\u0ccd\u0c95\u094b\u0cac\u094b\u0ca6\u0941 \u0c85\u0c82\u0ca4\u0cbe\u0c97\u0cbf \u0ca8\u0cbf\u0cb0\u0cbe\u0cb3 \u0c86\u0caf\u0ccd\u0ca4\u0941 \u0ca8\u0c82\u0c97\u0947.',
        'synth_text': '\u099a\u09c7\u09a8\u09cd\u09a8\u09be\u0987\u09df\u09c7\u09b0 \u09b6\u09c7\u09af\u09bc\u09be\u09b0\u09c7\u09b0 \u0985\u099f\u09cb\u09b0 \u09af\u09be\u09a4\u09cd\u09b0\u09c0\u09a6\u09c7\u09b0 \u09ae\u09a7\u09cd\u09af\u09c7 \u0996\u09be\u09ac\u09be\u09b0 \u09ad\u09be\u0997 \u0995\u09b0\u09c7 \u0996\u09be\u0993\u09df\u09be\u099f\u09be \u0986\u09ae\u09be\u09b0 \u0995\u09be\u099b\u09c7 \u09ae\u09a8 \u0996\u09c1\u09ac \u09ad\u09be\u09b2\u09cb \u0995\u09b0\u09c7 \u09a6\u09c7\u0993\u09df\u09be\u09b0 \u098f\u0995\u099f\u09bf \u09ac\u09bf\u09b7\u09af\u09bc\u0964',
    },
]

with gr.Blocks(title='IndicF5', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# \ud83c\udf0d IndicF5\n**Model:** `ai4bharat/IndicF5` (400M, bf16) \u00b7 **GPU:** {gpu_name} ({vram_gb:.0f} GB)\n\nNear-human polyglot TTS for 11 Indian languages. Zero-shot voice cloning from a 5–10s reference clip — upload your own, or pick one of the preloaded examples below.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Text to synthesize (use the target language\u2019s native script)',
                value=EXAMPLES[0]['synth_text'],
                lines=4,
                placeholder='Type in Hindi, Tamil, Bengali, Marathi, Punjabi, Telugu, etc.',
            )
            ref_audio = gr.Audio(
                label='Reference voice (5–10 seconds, any supported language)',
                type='filepath', info="5-10s in any of 11 Indian languages. The transcript must match exactly.",
                )
            ref_text = gr.Textbox(
                label='Reference transcript (in the reference audio\u2019s native script, language-accurate)',
                value=EXAMPLES[0]['ref_text'],
                lines=3,
            )
            btn = gr.Button('Generate speech', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated speech', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[[e['synth_text'], e['prompt'], e['ref_text']] for e in EXAMPLES],
        inputs=[text, ref_audio, ref_text],
        label='Preloaded voices (click to load) — each shows a cross-language example',
    )
    gr.Markdown('**Supported languages:** Assamese, Bengali, Gujarati, Hindi, Kannada, Malayalam, Marathi, Odia, Punjabi, Tamil, Telugu.\n\n**Tips:** The reference transcript must be in the **same language** as the reference audio and accurately capture what is spoken. IndicF5 only supports voice cloning — there are no fixed voices. For cloning, you must have permission from the speaker.')
    btn.click(ui_synth, inputs=[text, ref_audio, ref_text], outputs=[output_audio, status])

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ IndicF5 ready. Upload a ref clip + its transcript in any of 11 Indian languages.", outputs=[status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single IndicF5 generation from the notebook. Useful for scripting or quick checks.

TEXT = "\u0928\u092e\u0938\u094d\u0924\u0947! \u0938\u0902\u0917\u0940\u0924 \u0915\u0940 \u0924\u0930\u0939 \u091c\u0940\u0935\u0928 \u092d\u0940 \u0916\u0942\u092c\u0938\u0942\u0930\u0924 \u0939\u094b\u0924\u093e \u0939\u0948, \u092c\u0938 \u0907\u0938\u0947 \u0938\u0939\u0940 \u0924\u093e\u0932 \u092e\u0947\u0902 \u091c\u0940\u0928\u093e \u0906\u0928\u093e \u091a\u093e\u0939\u093f\u090f\u0964" # @param {type:"string"}
REF_AUDIO = "/content/ref_audio/PAN_F_HAPPY_00002.wav" # @param {type:"string"}
REF_TEXT = "\u0a07\u0a71\u0a15 \u0a17\u0a4d\u0a30\u0a3e\u0a39\u0a15 \u0a28\u0a47 \u0a38\u0a3e\u0a21\u0a40 \u0a2c\u0a47\u0a2e\u0a3f\u0a38\u0a3e\u0a32 \u0a38\u0a47\u0a35\u0a3e \u0a2c\u0a3e\u0a30\u0947 \u0a26\u0a3f\u0a32\u0a4b\u0a02\u0a17\u0a35\u0a3e\u0a39\u0a40 \u0a26\u0a3f\u0a71\u0a24\u0a40 \u0a1c\u0a3f\u0a38 \u0a28\u0a3e\u0a32 \u0a38\u0a3e\u0a28\u0a42\u0a02 \u0a05\u0a28\u0a70\u0a26 \u0a2e\u0a39\u0a3f\u0a38\u0a42\u0a38 \u0a39\u0a4b\u0a07\u0a06\u0961" # @param {type:"string"}

path, sr = synth(TEXT, REF_AUDIO, REF_TEXT)
print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of texts in the same target language. Each line in `BATCH` becomes one output file. Uses the same `REF_AUDIO` / `REF_TEXT` for the cloned voice across the batch.

BATCH_REF_AUDIO = '/content/ref_audio/PAN_F_HAPPY_00002.wav' # @param {type:"string"}
BATCH_REF_TEXT = "\u0a07\u0a71\u0a15 \u0a17\u0a4d\u0a30\u0a3e\u0a39\u0a15 \u0a28\u0a47 \u0a38\u0a3e\u0a21\u0a40 \u0a2c\u0a47\u0a2e\u0a3f\u0a38\u0a3e\u0a32 \u0a38\u0a47\u0a35\u0a3e \u0a2c\u0a3e\u0a30\u0947 \u0a26\u0a3f\u0a32\u0a4b\u0a02\u0a17\u0a35\u0a3e\u0a39\u0a40 \u0a26\u0a3f\u0a71\u0a24\u0a40 \u0a1c\u0a3f\u0a38 \u0a28\u0a3e\u0a32 \u0a38\u0a3e\u0a28\u0a42\u0a02 \u0a05\u0a28\u0a70\u0a26 \u0a2e\u0a39\u0a3f\u0a38\u0a42\u0a38 \u0a39\u0a4b\u0a07\u0a06\u0961" # @param {type:"string"}

BATCH = """\
\u0928\u092e\u0938\u094d\u0924\u0947, \u0906\u092a \u0915\u0948\u0938\u0947 \u0939\u0948\u0902?
\u092e\u0948\u0902 \u0925\u094b\u0921\u093c\u093e \u0905\u091a\u094d\u091b\u093e, \u0927\u0928\u094d\u092f\u0935\u093e\u0926\u0964
\u0906\u091c \u092e\u094c\u0938\u092e \u0915\u0947 \u0926\u093f\u0928 \u092c\u0939\u0941\u0924 \u0920\u0902\u0921 \u0939\u0948, \u0915\u094d\u092f\u093e \u0906\u092a\u0928\u0947 \u092a\u094d\u0932\u093e\u0928 \u0915\u093f\u092f\u093e \u0939\u0948?
\u092e\u0941\u091d\u0947 \u090f\u0915 \u0915\u092a \u091a\u093e\u092f \u091a\u093e\u0939\u093f\u090f, \u0925\u094b\u0921\u093c\u093e \u091c\u0932\u0926\u0940 \u092e\u093f\u0932\u093e\u0901\u0917\u0940\u0964
"""

lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)

for i, line in enumerate(lines):
    try:
        print(f'[{i+1}/{len(lines)}] {line[:60]}{"..." if len(line) > 60 else ""}')
        path, sr = synth(line, BATCH_REF_AUDIO, BATCH_REF_TEXT, out_name=f'{i:03d}.wav')
        os.rename(path, os.path.join(out_dir, f'{i:03d}.wav'))

    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f'\n[Done] {len(lines)} files written to {out_dir}')